In [2]:
import importlib
import RAG_model.ingestion.config as config

print(config.__file__)  # confirms which config.py Python loaded

importlib.reload(config)

print(config.GENERATION_MODEL)

ModuleNotFoundError: No module named 'RAG_model'

In [3]:
from pathlib import Path
import sys

PROJECT_ROOT = Path(r"C:\Users\david\Documents\Projects\crypto_chatbot_rag")
SRC_PATH = PROJECT_ROOT / "src"

assert (SRC_PATH / "RAG_model").is_dir(), f"Could not find package at: {SRC_PATH}"

sys.path.insert(0, str(SRC_PATH))

print("Added to Python path:", SRC_PATH)

Added to Python path: C:\Users\david\Documents\Projects\crypto_chatbot_rag\src


In [4]:
from RAG_model.ingestion.config import EMBEDDINGS_MODEL, COLLECTION_NAME, DB_PATH_NAME, GENERATION_MODEL
from RAG_model.ingestion.embedding import create_openrouter_client
from qdrant_client import QdrantClient
from tenacity import retry, wait_exponential, stop_after_attempt
from time import perf_counter

In [5]:
wait = wait_exponential(multiplier=1, min=10, max=240)


In [6]:
qdrant_client = QdrantClient(path = str(DB_PATH_NAME))


In [7]:
RETRIEVAL_K = 5

In [8]:
client = create_openrouter_client()

In [9]:
def fetch_context(question, retrieval_k = RETRIEVAL_K):
    """Embed a question and retrieve its top-k most similar chunks."""
    
    # Embedd the question with the same embedding model
    query = client.embeddings.create(model = EMBEDDINGS_MODEL, input=[question]).data[0].embedding
    
    # Return results from the database based on the question
    results = qdrant_client.query_points(
        collection_name= COLLECTION_NAME,
        query = query,
        limit= retrieval_k,
        with_payload= True)
        
    # Convert the payload to the right format 
    
    chunks = []
    
    for point in results.points:
        payload = point.payload or {}

        chunks.append(
            {
                "chunk_id": payload["chunk_id"],
                "chunk_text": payload["chunk_text"],
                "ticker": payload["ticker"],
                "filing_date": payload["filing_date"],
                "section_title": payload["chunk_title"],
                "source_url": payload["source_url"],
                "score": point.score,
            }
        )

    return chunks

In [10]:
results = fetch_context("What is FETH?")
for rank, result in enumerate(results, start=1):
    print(f"\n--- Result {rank} ---")
    print(f"Score: {result['score']:.4f}")
    print(f"Chunk ID: {result['chunk_id']}")
    print(f"Ticker: {result['ticker']}")
    print(f"Section: {result['section_title']}")
    print("Result:")
    print(result["chunk_text"][:600])


--- Result 1 ---
Score: 0.5439
Chunk ID: 0000950170-25-039374*section-5*table-0015
Ticker: FETH
Section: Market for Registrant’s Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities
Result:
Section: Market for Registrant’s Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities
Table ID: table-0015

Trust | Commencement of Operations | Ticker Symbol | Name of each exchange on which registered
Fidelity Ethereum Fund | July 23, 2024 | FETH | Cboe BZX Exchange, Inc.

--- Result 2 ---
Score: 0.5409
Chunk ID: 0001193125-26-071486*section-5*table-0022
Ticker: FETH
Section: Market for Registrant’s Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities
Result:
Section: Market for Registrant’s Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities
Table ID: table-0022

Trust | Commencement of Operations | Ticker Symbol | Name of each exchange on which registered
Fidelity Eth

In [11]:
def make_rag_messages(question, history, chunks):
    """Build chat messages for the RAG answer step: system (with context) + history + user question."""
    context = "\n\n".join(
        f"-Extract from:\n \
           [SOURCE:{chunk['chunk_id']:}]\n \
            - TICKER: {chunk['ticker']}\n \
            - Section:{chunk['section_title']}\n \
            - Filing Date:{chunk['filing_date']}\n \
            - SEC URL{chunk['source_url']}] \n \
            - Content: \n {chunk['chunk_text']}  " for chunk in chunks)
    
    system_prompt = f"""
    You are a knowledgeable, friendly assistant that helps with question regard some specific companies extracting only the information available from the SEC 10-K and 10-Q forms those companies submmit.
    You are chatting with a user about Answer questions about these SEC filing entities: IBIT, ETHA, FBTC, FETH, GBTC, ETHE. If the question is outside this corpus, explain that the corpus does
    not contain evidence to answer it.
    Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
    Answer using only the supplied context.

    If the context does not contain enough information, say:
    "I could not find enough evidence in the retrieved SEC filings."

    Cite the supplied source IDs for every factual claim using:
    [SOURCE: chunk_id]

    Never invent a source ID or cite a source that was not supplied.
    
    Do not infer, calculate, or compare values unless the retrieved context
    contains all evidence needed. Otherwise say that there is not enough evidence.
    
    For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
    {context}

    With this context, please answer the user's question. Be accurate, relevant and complete.
    """
    
    return (
        [{"role": "system",
          "content": system_prompt}]
        + history
        + [{"role": "user",
            "content": question}]
        )

In [12]:
@retry(wait=wait,
       stop=stop_after_attempt(4))
def answer(question: str, history:list[dict] | None = None):
    
    if history is None:
        history = []
    
    total_start = perf_counter()
    
    
    retrieval_start = perf_counter()
    chunks = fetch_context(question)
    retrieval_end = perf_counter()
    
    prompt_start = perf_counter()
    messages = make_rag_messages(question, history, chunks)
    prompt_end = perf_counter()
    
    generation_start = perf_counter()
    response = client.chat.completions.create(
        model=GENERATION_MODEL,
        messages=messages,
    )
    generation_end = perf_counter()
    
    answer_text = response.choices[0].message.content
    
    latency_ms = {
        "retrieval": (retrieval_end - retrieval_start) * 1000,
        "prompt_building": (prompt_end - prompt_start) * 1000,
        "generation": (generation_end - generation_start) * 1000,
        "total": (generation_end - total_start) * 1000,
    }
    
    return question, answer_text, chunks, latency_ms
    

In [32]:
def format_answer(answer):
    
    question, answer_text, chunks, latency_ms = answer
    
    scores = [ {"chunk_id": chunk["chunk_id"],
                "scores": chunk["score"]} for chunk in chunks]
    
    return {
        "User Question": question,
        "Answer": answer_text,
        "Similarity Scores": scores,
        "Retrieved Chunk texts": chunks,
        "Citations": answer_text.split("SOURCE")[1],
        "Latency": latency_ms
    }
    
    

In [33]:
answer_text = answer("What is the ticker symbol for Fidelity Ethereum Fund?")

In [31]:
print(answer_text)

('What is the ticker symbol for Fidelity Ethereum Fund?', 'The ticker symbol for Fidelity Ethereum Fund is FETH [SOURCE:0000950170-25-039374*section-5*table-0015].', [{'chunk_id': '0000950170-25-039374*section-5*table-0015', 'chunk_text': 'Section: Market for Registrant’s Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities\nTable ID: table-0015\n\nTrust | Commencement of Operations | Ticker Symbol | Name of each exchange on which registered\nFidelity Ethereum Fund | July 23, 2024 | FETH | Cboe BZX Exchange, Inc.', 'ticker': 'FETH', 'filing_date': '2025-03-14', 'section_title': 'Market for Registrant’s Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities', 'source_url': 'https://www.sec.gov/Archives/edgar/data/2000046/000095017025039374/ck0002000046-20241231.htm', 'score': 0.7696122660117776}, {'chunk_id': '0001193125-26-071486*section-5*table-0022', 'chunk_text': 'Section: Market for Registrant’s Common Equity, Related St

In [34]:
formatted = format_answer (answer_text)
print(formatted)

{'User Question': 'What is the ticker symbol for Fidelity Ethereum Fund?', 'Answer': 'The ticker symbol for Fidelity Ethereum Fund is FETH [SOURCE:0000950170-25-039374*section-5*table-0015].', 'Similarity Scores': [{'chunk_id': '0000950170-25-039374*section-5*table-0015', 'scores': 0.7696122660117776}, {'chunk_id': '0001193125-26-071486*section-5*table-0022', 'scores': 0.7674688930528025}, {'chunk_id': '0000950170-25-039374*section-1*text-0000', 'scores': 0.7344514207047053}, {'chunk_id': '0001193125-26-071486*section-8*text-0005', 'scores': 0.7327740088551105}, {'chunk_id': '0001193125-26-071486*section-1*text-0000', 'scores': 0.7319195841266679}], 'Retrieved Chunk texts': [{'chunk_id': '0000950170-25-039374*section-5*table-0015', 'chunk_text': 'Section: Market for Registrant’s Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities\nTable ID: table-0015\n\nTrust | Commencement of Operations | Ticker Symbol | Name of each exchange on which registered\nFide

In [ ]:
print(metrics)

{'retrieval': 283.0915999948047, 'prompt_building': 0.04169999738223851, 'generation': 1810.5744000058621, 'total': 2093.708100001095}


In [17]:
print(chunks)

[{'chunk_id': '0000950170-25-039374*section-5*table-0015', 'chunk_text': 'Section: Market for Registrant’s Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities\nTable ID: table-0015\n\nTrust | Commencement of Operations | Ticker Symbol | Name of each exchange on which registered\nFidelity Ethereum Fund | July 23, 2024 | FETH | Cboe BZX Exchange, Inc.', 'ticker': 'FETH', 'filing_date': '2025-03-14', 'section_title': 'Market for Registrant’s Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities', 'source_url': 'https://www.sec.gov/Archives/edgar/data/2000046/000095017025039374/ck0002000046-20241231.htm', 'score': 0.7696242118262883}, {'chunk_id': '0001193125-26-071486*section-5*table-0022', 'chunk_text': 'Section: Market for Registrant’s Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities\nTable ID: table-0022\n\nTrust | Commencement of Operations | Ticker Symbol | Name of each exchange on whi

In [ ]:
client.close()
qdrant_client.close()
print("Qdrant client closed.")

Qdrant client closed.
